# Candida intermedia FluxFormer evaluation

Evaluate a trained FluxFormer against the separately generated **pFBA test dataset**.

- Inputs: 17 signed exchange lower bounds; targets: 1,971 signed net reaction fluxes.
- Plot training/validation loss and five representative Candida flux diagnostics.
- Finish with pooled **R², MAE, and RMSE** across every test row and all 1,971 fluxes, including the 17 injected exchange slots.

Run from the repository root. Set the checkpoint and test-data paths below. All test rows are evaluated; the test file is not split again. Metrics describe fidelity to pFBA, not experimental growth accuracy.

In [ ]:
from datetime import date
import json
from pathlib import Path
import time
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from flux_transformer import FluxTransformer
from generate_candida_intermedia_data import DEFAULT_MODEL, INPUT_EXCHANGES, file_sha256

# Set these to the completed Candida training checkpoint and independent test dataset.
model_name = "candida_intermedia_100k_signed_pfba_d128_h8_l3_ff640"
CHECKPOINT_PATH = Path("models") / model_name / f"{model_name}_checkpoint.pth"
DATA_PATH = Path("data/2026-11-02_candida_intermedia_test_data_50000_samples.csv")
METADATA_PATH = DATA_PATH.with_suffix(".json")
MODEL_PATH = DEFAULT_MODEL

INFERENCE_BATCH_SIZE = 4  # Full-vocabulary attention; reduce this if GPU memory is limited.
CSV_CHUNK_SIZE = 512
SAVE_PLOTS = False
pic_dir = Path("pics") / date.today().isoformat() / model_name
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
print(f"Device: {device}")

## Load the trained model

Use the shared injected-token architecture and the checkpoint's saved dimensions and reaction ordering. Both ordinary-Huber and normalized-Huber trainer checkpoints use this format; predictions remain in the original flux scale.

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
config = checkpoint["config"]
data_info = checkpoint["data_info"]
input_cols = list(data_info["input_cols"])
outputs = list(data_info["output_cols"])
input_token_indices = [int(i) for i in data_info["input_token_indices"]]
if input_cols != list(INPUT_EXCHANGES):
    raise ValueError("Checkpoint input names/order differ from the 17-exchange Candida pilot.")
if len(outputs) != 1971 or int(config["vocab_size"]) != len(outputs):
    raise ValueError("Expected the full 1,971-reaction Candida vocabulary.")
if input_token_indices != [outputs.index(f"{name}_flux") for name in input_cols]:
    raise ValueError("Checkpoint input-to-flux token mapping is inconsistent.")
if list(config["input_token_indices"]) != input_token_indices:
    raise ValueError("Checkpoint config and data_info disagree on injected token indices.")
model = FluxTransformer(
    vocab_size=int(config["vocab_size"]), input_token_indices=input_token_indices,
    d_model=int(config["d_model"]), n_heads=int(config["n_heads"]),
    n_layers=int(config["n_layers"]), d_ff=int(config["d_ff"]),
    dropout=float(config["dropout"]),
)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
if model.input_token_indices.tolist() != input_token_indices:
    raise ValueError("Saved model buffer differs from checkpoint token metadata.")
model = model.to(device).eval()
train_losses = list(checkpoint.get("train_losses", []))
validation_losses = list(checkpoint.get("test_losses", []))
del checkpoint
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Vocabulary: {len(outputs):,} reactions | Injected inputs: {len(input_cols)}")
print(f"Architecture: d={config['d_model']}, heads={config['n_heads']}, "
      f"layers={config['n_layers']}, ff={config['d_ff']}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Training rows: {data_info['n_train']:,} | Validation rows: {data_info['n_test']:,}")

## Load test-data information

Check the CSV schema and generation sidecar before inference. Signed uptake bounds are injected unchanged into their reaction tokens; they are constraints, not achieved uptake fluxes.

In [ ]:
test_metadata = json.loads(METADATA_PATH.read_text())
if test_metadata["status"] != "complete" or test_metadata["arguments"]["dataset"] != "test":
    raise ValueError("Use a completed, separately generated test dataset.")
if test_metadata["arguments"]["flux_solver_mode"] != "pfba":
    raise ValueError("This notebook expects pFBA-generated test targets.")
if test_metadata["input_columns"] != input_cols or test_metadata["output_columns"] != outputs:
    raise ValueError("Test-data token names/order differ from the checkpoint.")
if test_metadata["model_sha256"] != file_sha256(MODEL_PATH):
    raise ValueError("Test data were generated from a different SBML model file.")
csv_columns = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
if csv_columns != input_cols + outputs:
    raise ValueError("CSV columns do not match the checkpoint's input/output schema.")
reaction_ids = [name.removesuffix("_flux") for name in outputs]
sbml_root = ET.parse(MODEL_PATH).getroot()
sbml_reactions = sbml_root.findall(".//{*}listOfReactions/{*}reaction")
reaction_names = {
    reaction.attrib["id"].removeprefix("R_"): reaction.attrib.get("name", "")
    for reaction in sbml_reactions
}
if set(reaction_ids) != set(reaction_names):
    raise ValueError("Checkpoint reaction vocabulary differs from the supplied Candida model.")
print(f"Test data: {DATA_PATH}")
print(f"Rows: {test_metadata['accepted_samples']:,} | Generation seed: {test_metadata['arguments']['seed']}")
print(f"pFBA fraction of optimum: {test_metadata['arguments']['pfba_fraction_of_optimum']}")
print("All test rows will be evaluated, with signed inputs and targets preserved.")

## Training and validation loss

The checkpoint calls its held-out training-split history `test_losses`; here it is labeled **validation**. The independent test CSV above was not used to produce these curves.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(np.arange(1, len(train_losses) + 1), train_losses, label="Training")
ax.plot(np.arange(1, len(validation_losses) + 1), validation_losses, label="Validation")
if train_losses and validation_losses and min(train_losses + validation_losses) > 0:
    ax.set_yscale("log")
ax.set(xlabel="Epoch", ylabel="Loss", title=f"Training and validation loss: {model_name}")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
if SAVE_PLOTS:
    pic_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(pic_dir / f"{model_name}_loss.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

## Predict the independent test set

Stream the CSV and predict with **all reaction tokens present** in each forward pass. Keep the five diagnostic fluxes and accumulate errors for every output, avoiding a full test-set attention pass or repeated inference for each plot.

Selected reactions: biomass; cytosolic phosphoglycerate kinase and ATP-dependent pyruvate kinase (glycolysis); cytosolic glucose-6-phosphate dehydrogenase (pentose phosphate pathway); and mitochondrial citrate synthase (TCA cycle).

In [ ]:
diagnostic_fluxes = {
    "BIOMASS_C_flux": "Biomass",
    "UF01405_C_flux": "Phosphoglycerate kinase (cytosol)",
    "UF01344_C_flux": "Pyruvate kinase, ATP-dependent (cytosol)",
    "UF00256_C_flux": "Glucose-6-phosphate dehydrogenase (cytosol)",
    "UF01062_M_flux": "Citrate synthase (mitochondrion)",
}
diagnostic_indices = [outputs.index(name) for name in diagnostic_fluxes]
if test_metadata["objective"] != {"BIOMASS_C": 1.0}:
    raise ValueError("Review the biomass diagnostic: the test-data objective changed.")
if INFERENCE_BATCH_SIZE < 1 or CSV_CHUNK_SIZE < 1:
    raise ValueError("Batch and CSV chunk sizes must be positive.")

diagnostic_true_parts, diagnostic_pred_parts = [], []
pooled_stats = dict(n_values=0, n_rows=0, true_sum=0.0, true_square_sum=0.0,
                    absolute_error_sum=0.0, squared_error_sum=0.0)
start_time = time.monotonic()
next_progress = 5000
with torch.inference_mode():
    for chunk in pd.read_csv(DATA_PATH, dtype=np.float32, chunksize=CSV_CHUNK_SIZE):
        input_values = chunk[input_cols].to_numpy()
        target_values = chunk[outputs].to_numpy()
        if not np.isfinite(input_values).all() or not np.isfinite(target_values).all():
            raise ValueError("Test data contain nonfinite inputs or targets.")
        if (input_values > 0).any():
            raise ValueError("Expected signed nonpositive uptake lower bounds.")
        for begin in range(0, len(chunk), INFERENCE_BATCH_SIZE):
            end = min(begin + INFERENCE_BATCH_SIZE, len(chunk))
            token_inputs = torch.zeros((end - begin, len(outputs), 1), dtype=torch.float32, device=device)
            token_inputs[:, input_token_indices, 0] = torch.as_tensor(input_values[begin:end], device=device)
            predictions, selected_indices = model(token_inputs, output_subset=None)
            if not torch.equal(selected_indices.cpu(), torch.arange(len(outputs))):
                raise ValueError("Forward-pass output token order differs from the saved schema.")
            prediction_values = predictions[:, :, -1].cpu().numpy().astype(np.float64)
            truth_values = target_values[begin:end].astype(np.float64)
            if not np.isfinite(prediction_values).all():
                raise ValueError("Model produced nonfinite predictions.")
            errors = truth_values - prediction_values
            pooled_stats["n_values"] += truth_values.size
            pooled_stats["n_rows"] += len(truth_values)
            pooled_stats["true_sum"] += float(truth_values.sum())
            pooled_stats["true_square_sum"] += float(np.square(truth_values).sum())
            pooled_stats["absolute_error_sum"] += float(np.abs(errors).sum())
            pooled_stats["squared_error_sum"] += float(np.square(errors).sum())
            diagnostic_true_parts.append(truth_values[:, diagnostic_indices].copy())
            diagnostic_pred_parts.append(prediction_values[:, diagnostic_indices].copy())
            del predictions, token_inputs
        if pooled_stats["n_rows"] >= next_progress:
            print(f"Predicted {pooled_stats['n_rows']:,} / {test_metadata['accepted_samples']:,} rows")
            next_progress += 5000
if pooled_stats["n_rows"] != test_metadata["accepted_samples"] or pooled_stats["n_rows"] < 1:
    raise ValueError("CSV row count differs from its generation sidecar, or the test set is empty.")
diagnostic_y_true = np.concatenate(diagnostic_true_parts)
diagnostic_y_pred = np.concatenate(diagnostic_pred_parts)
del diagnostic_true_parts, diagnostic_pred_parts
print(f"Predicted all {pooled_stats['n_rows']:,} test conditions in {time.monotonic() - start_time:.1f} s.")

## Flux diagnostic plots

For each of the five fluxes, show true versus predicted values, residuals, the prediction-error distribution, and the target distribution. All test conditions are included; signs and original model units are preserved.

In [ ]:
def plot_diagnostics_2x2(y_true, y_pred, label, save_path=None):
    residuals = y_true - y_pred
    max_error = max(float(np.abs(residuals).max()), np.finfo(float).eps)
    lo, hi = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
    if lo == hi:
        padding = max(abs(float(lo)) * 0.05, 0.01)
        lo, hi = lo - padding, hi + padding
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    axes[0, 0].scatter(y_true, y_pred, s=5, alpha=0.2, color="royalblue", rasterized=True)
    axes[0, 0].plot([lo, hi], [lo, hi], "k--", lw=1.5)
    axes[0, 0].set(title="True vs predicted", xlabel="True flux", ylabel="Predicted flux")
    axes[0, 1].scatter(y_true, residuals, s=5, alpha=0.2, color="darkorange", rasterized=True)
    axes[0, 1].axhline(0, color="red", lw=1)
    axes[0, 1].set(title="Residuals", xlabel="True flux", ylabel="True - predicted",
                   ylim=(-max_error, max_error))
    sns.histplot(residuals, bins=60, kde=bool(np.ptp(residuals) > 0), stat="density",
                 color="indianred", ax=axes[1, 0])
    axes[1, 0].set(title="Prediction-error distribution", xlabel="True - predicted", ylabel="Density")
    sns.histplot(y_true, bins=60, kde=bool(np.ptp(y_true) > 0), stat="density",
                 color="mediumseagreen", ax=axes[1, 1])
    axes[1, 1].set(title="True-flux distribution", xlabel="True flux", ylabel="Density")
    for ax in axes.ravel():
        ax.grid(True, alpha=0.25)
    fig.suptitle(label, fontsize=14)
    fig.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

for column, (flux_name, description) in enumerate(diagnostic_fluxes.items()):
    save_path = pic_dir / f"{model_name}_{flux_name}_diagnostics.png" if SAVE_PLOTS else None
    plot_diagnostics_2x2(
        diagnostic_y_true[:, column], diagnostic_y_pred[:, column],
        label=f"{description} | {flux_name.removesuffix('_flux')}", save_path=save_path,
    )

## Overall pooled metrics

Pool every sample–reaction pair into one evaluation population, including the 17 injected exchange slots. These are **not averages of per-reaction metrics** and are computed in the original signed flux scale. Higher-magnitude fluxes influence pooled R²/RMSE more strongly. The training loss excludes injected tokens; this requested overall evaluation includes them.

In [ ]:
# Final cell: pooled metrics across every test sample and all 1,971 flux outputs.
n_values = pooled_stats["n_values"]
overall_mae = pooled_stats["absolute_error_sum"] / n_values
overall_rmse = np.sqrt(pooled_stats["squared_error_sum"] / n_values)
total_sum_squares = max(
    0.0, pooled_stats["true_square_sum"] - pooled_stats["true_sum"] ** 2 / n_values
)
if total_sum_squares > 0:
    overall_r2 = 1.0 - pooled_stats["squared_error_sum"] / total_sum_squares
else:
    overall_r2 = 1.0 if pooled_stats["squared_error_sum"] == 0 else 0.0
pooled_metrics = {"R2": float(overall_r2), "MAE": float(overall_mae), "RMSE": float(overall_rmse)}
print(f"Overall pooled metrics: {pooled_stats['n_rows']:,} samples × {len(outputs):,} fluxes")
print(f"R²   : {overall_r2:.6f}")
print(f"MAE  : {overall_mae:.6f}")
print(f"RMSE : {overall_rmse:.6f}")